# 🏋️ Notebook 2 — Entraînement du Modèle SwinIR

## Restauration d'images : Débruitage et Super-Résolution

---

### Objectif de ce notebook

Ce notebook implémente l'entraînement du modèle **SwinIR** (Swin Transformer for Image Restoration) sur le dataset **DIV2K**.

SwinIR est une architecture basée sur le **Swin Transformer** qui atteint des performances état-de-l'art pour la super-résolution et le débruitage d'images.

**Plan du notebook :**
1. Configuration et installation des dépendances
2. Définition de l'architecture SwinIR
3. Préparation du dataset (paires HR/LR)
4. Pipeline d'entraînement
5. Suivi des métriques et sauvegarde des checkpoints
6. Visualisation des courbes d'entraînement

## 1. Configuration de l'environnement

On installe les dépendances nécessaires et on configure PyTorch pour l'entraînement sur GPU (Kaggle fournit un GPU T4 ou P100).  
L'utilisation du GPU est indispensable pour entraîner un modèle de type Transformer en temps raisonnable.

In [ ]:
# Installation des dépendances
!pip install timm einops --quiet

In [ ]:
import os
import math
import json
import random
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from einops import rearrange

import warnings
warnings.filterwarnings('ignore')

# ── Chemins ──────────────────────────────────────────────────────────────────
DATASET_PATH  = "/kaggle/input/datasets/soumikrakshit/div2k-high-resolution-images/DIV2K_train_HR/DIV2K_train_HR"
RESULTS_DIR   = Path("results/training")
CKPT_DIR      = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── GPU ──────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device utilisé : {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM disponible : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── Seed ─────────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
print("✅ Environnement configuré.")

## 2. Hyperparamètres d'entraînement

On centralise tous les hyperparamètres pour faciliter les expériences.  
- **SCALE** : facteur d'agrandissement (×4 est le benchmark standard dans DIV2K)
- **PATCH_SIZE** : taille des patches extraits lors de l'entraînement (compromise entre contexte et mémoire)
- **BATCH_SIZE** : adapté à la VRAM disponible (16 GB sur P100)

In [ ]:
# ════════════════════════════════════════════════════════════════════
#                    HYPERPARAMÈTRES
# ════════════════════════════════════════════════════════════════════
CONFIG = {
    # Dataset
    'scale'            : 4,          # Facteur de super-résolution (×4)
    'patch_size_hr'    : 256,        # Taille du patch HR (LR = 256/scale)
    'val_split'        : 0.1,        # 10% pour validation

    # Modèle SwinIR
    'img_size'         : 64,         # Taille d'entrée du transformer (patch LR)
    'window_size'      : 8,
    'embed_dim'        : 60,         # Dimension des embeddings (modèle léger)
    'depths'           : [6, 6, 6, 6],
    'num_heads'        : [6, 6, 6, 6],
    'mlp_ratio'        : 2.0,
    'upscale'          : 4,
    'in_chans'         : 3,

    # Entraînement
    'num_epochs'       : 50,
    'batch_size'       : 8,
    'lr'               : 2e-4,
    'lr_decay_epochs'  : [25, 40],
    'lr_decay_gamma'   : 0.5,
    'weight_decay'     : 1e-4,
    'num_workers'      : 2,
    'save_every'       : 10,         # Sauvegarder un checkpoint toutes les N époques
    'log_every'        : 20,         # Afficher les stats tous les N batches
}

print("📋 Configuration de l'entraînement :")
for k, v in CONFIG.items():
    print(f"   {k:<25} : {v}")

## 3. Architecture SwinIR

SwinIR repose sur le **Swin Transformer** avec les modules clés suivants :
- **Shallow Feature Extraction** : une convolution initiale pour extraire les features bas niveau
- **Residual Swin Transformer Blocks (RSTB)** : blocs profonds avec attention locale par fenêtres glissantes
- **Image Reconstruction** : module de reconstruction avec PixelShuffle pour l'upscaling

L'attention par fenêtres (**Window Self-Attention**) permet de modéliser les dépendances locales efficacement, tandis que les fenêtres décalées (**Shifted Windows**) permettent les interactions entre fenêtres.

In [ ]:
# ════════════════════════════════════════════════════════════════════
#          ARCHITECTURE SWINIR (version simplifiée / légère)
# ════════════════════════════════════════════════════════════════════

class WindowAttention(nn.Module):
    """Multi-head self-attention pour fenêtres locales."""
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        # Biais de position relatif
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) * (2 * window_size - 1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)

        coords_h = torch.arange(self.window_size)
        coords_w = torch.arange(self.window_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += self.window_size - 1
        relative_coords[:, :, 1] += self.window_size - 1
        relative_coords[:, :, 0] *= 2 * self.window_size - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer('relative_position_index', relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)].view(
            self.window_size * self.window_size, self.window_size * self.window_size, -1)
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


def window_partition(x, window_size):
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class SwinTransformerLayer(nn.Module):
    """Un seul bloc Swin Transformer avec ou sans décalage de fenêtre."""
    def __init__(self, dim, num_heads, window_size=7, shift_size=0, mlp_ratio=4.,
                 drop=0., attn_drop=0.):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size=window_size, num_heads=num_heads, attn_drop=attn_drop)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(drop),
        )

    def forward(self, x, H, W):
        B, L, C = x.shape
        assert L == H * W
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # Padding
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        if pad_r > 0 or pad_b > 0:
            x = F.pad(x, (0, 0, 0, pad_r, 0, pad_b))
        _, Hp, Wp, _ = x.shape

        # Shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        # Window partition
        x_windows = window_partition(shifted_x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        attn_windows = self.attn(x_windows)

        # Reverse window
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, Hp, Wp)

        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()

        x = x.view(B, H * W, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class RSTB(nn.Module):
    """Residual Swin Transformer Block."""
    def __init__(self, dim, depth, num_heads, window_size, mlp_ratio=4.):
        super().__init__()
        self.layers = nn.ModuleList([
            SwinTransformerLayer(
                dim=dim, num_heads=num_heads, window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio
            ) for i in range(depth)
        ])
        self.conv = nn.Conv2d(dim, dim, 3, 1, 1)

    def forward(self, x, H, W):
        res = x
        for layer in self.layers:
            x = layer(x, H, W)
        x = x.permute(0, 2, 1).view(-1, x.shape[-1], H, W).contiguous()
        x = self.conv(x)
        x = x.flatten(2).transpose(1, 2)
        return x + res


class SwinIR(nn.Module):
    """
    SwinIR : Image Restoration using Swin Transformer
    Architecture légère adaptée à l'entraînement sur Kaggle.
    """
    def __init__(self, img_size=64, in_chans=3, embed_dim=60,
                 depths=(6, 6, 6, 6), num_heads=(6, 6, 6, 6),
                 window_size=8, mlp_ratio=2., upscale=4):
        super().__init__()
        self.window_size = window_size
        self.upscale = upscale
        self.img_size = img_size

        # ── 1. Shallow feature extraction ────────────────────────────────────
        self.conv_first = nn.Conv2d(in_chans, embed_dim, 3, 1, 1)

        # ── 2. Deep feature extraction (RSTB blocks) ─────────────────────────
        self.num_layers = len(depths)
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            self.layers.append(RSTB(
                dim=embed_dim, depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size, mlp_ratio=mlp_ratio
            ))
        self.norm = nn.LayerNorm(embed_dim)
        self.conv_after_body = nn.Conv2d(embed_dim, embed_dim, 3, 1, 1)

        # ── 3. Upsampling + Reconstruction ───────────────────────────────────
        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, (upscale ** 2) * in_chans, 3, 1, 1),
            nn.PixelShuffle(upscale),
        )
        self.conv_last = nn.Conv2d(in_chans, in_chans, 3, 1, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.zeros_(m.bias)
                nn.init.ones_(m.weight)
            elif isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        H, W = x.shape[2], x.shape[3]

        # 1. Shallow features
        x = self.conv_first(x)  # B, C, H, W
        res = x

        # 2. Deep features
        B, C, H_x, W_x = x.shape
        x_seq = x.flatten(2).transpose(1, 2)  # B, H*W, C
        for layer in self.layers:
            x_seq = layer(x_seq, H_x, W_x)
        x_seq = self.norm(x_seq)
        x_feat = x_seq.transpose(1, 2).view(B, C, H_x, W_x)
        x = self.conv_after_body(x_feat) + res

        # 3. Upsampling
        x = self.upsample(x)
        x = self.conv_last(x)
        return x

# Test du modèle
model = SwinIR(
    img_size=CONFIG['img_size'],
    in_chans=CONFIG['in_chans'],
    embed_dim=CONFIG['embed_dim'],
    depths=CONFIG['depths'],
    num_heads=CONFIG['num_heads'],
    window_size=CONFIG['window_size'],
    mlp_ratio=CONFIG['mlp_ratio'],
    upscale=CONFIG['upscale'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Modèle SwinIR créé.")
print(f"   Paramètres entraînables : {n_params:,} ({n_params/1e6:.2f}M)")

# Test forward pass
with torch.no_grad():
    dummy = torch.randn(1, 3, CONFIG['patch_size_hr'] // CONFIG['scale'],
                        CONFIG['patch_size_hr'] // CONFIG['scale']).to(DEVICE)
    out = model(dummy)
print(f"   Input LR shape  : {tuple(dummy.shape)}")
print(f"   Output HR shape : {tuple(out.shape)}")

## 4. Préparation du Dataset

On crée des paires **(LR, HR)** à partir des images DIV2K :  
- **HR** : patch découpé dans l'image originale haute résolution (256×256)
- **LR** : version dégradée par sous-échantillonnage bicubique ×4 (64×64)

### Augmentations appliquées :
- Flip horizontal et vertical aléatoires
- Rotation 90°
Ces augmentations augmentent artificiellement la diversité et améliorent la robustesse du modèle.

In [ ]:
class DIV2KDataset(Dataset):
    """
    Dataset DIV2K pour la super-résolution.
    Génère des paires (LR, HR) par sous-échantillonnage bicubique.
    """
    def __init__(self, image_paths, patch_size_hr=256, scale=4, augment=True):
        self.image_paths = image_paths
        self.patch_size_hr = patch_size_hr
        self.patch_size_lr = patch_size_hr // scale
        self.scale = scale
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        w, h = img.size

        # S'assurer que l'image est assez grande
        if w < self.patch_size_hr or h < self.patch_size_hr:
            img = img.resize(
                (max(w, self.patch_size_hr), max(h, self.patch_size_hr)),
                Image.BICUBIC
            )
            w, h = img.size

        # Extraction aléatoire d'un patch HR
        x = random.randint(0, w - self.patch_size_hr)
        y = random.randint(0, h - self.patch_size_hr)
        hr_patch = img.crop((x, y, x + self.patch_size_hr, y + self.patch_size_hr))

        # Création du patch LR par sous-échantillonnage bicubique
        lr_patch = hr_patch.resize(
            (self.patch_size_lr, self.patch_size_lr), Image.BICUBIC
        )

        # Conversion en tenseurs
        hr = TF.to_tensor(hr_patch)  # [0, 1]
        lr = TF.to_tensor(lr_patch)

        # Augmentations (appliquées de manière cohérente sur HR et LR)
        if self.augment:
            if random.random() > 0.5:
                hr = TF.hflip(hr)
                lr = TF.hflip(lr)
            if random.random() > 0.5:
                hr = TF.vflip(hr)
                lr = TF.vflip(lr)
            k = random.randint(0, 3)
            if k > 0:
                hr = torch.rot90(hr, k, dims=[1, 2])
                lr = torch.rot90(lr, k, dims=[1, 2])

        return lr, hr


# Chargement et split train/val
extensions = ['*.png', '*.jpg', '*.jpeg']
all_images = []
for ext in extensions:
    all_images.extend(glob.glob(os.path.join(DATASET_PATH, ext)))
all_images = sorted(all_images)

random.shuffle(all_images)
n_val  = max(1, int(len(all_images) * CONFIG['val_split']))
n_train = len(all_images) - n_val
train_paths = all_images[:n_train]
val_paths   = all_images[n_train:]

train_dataset = DIV2KDataset(train_paths, CONFIG['patch_size_hr'], CONFIG['scale'], augment=True)
val_dataset   = DIV2KDataset(val_paths,   CONFIG['patch_size_hr'], CONFIG['scale'], augment=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True,  num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                          shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

print(f"✅ Dataset préparé :")
print(f"   Train : {len(train_dataset)} images → {len(train_loader)} batches")
print(f"   Val   : {len(val_dataset)} images   → {len(val_loader)} batches")

# Vérification visuelle d'une paire
lr_sample, hr_sample = train_dataset[0]
print(f"   LR shape : {lr_sample.shape}, HR shape : {hr_sample.shape}")

In [ ]:
# Visualisation d'une paire LR/HR d'exemple
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Exemple de paire LR → HR (avant entraînement)', fontsize=12, fontweight='bold')

lr_np = lr_sample.permute(1, 2, 0).numpy()
hr_np = hr_sample.permute(1, 2, 0).numpy()

axes[0].imshow(lr_np)
axes[0].set_title(f'LR (Basse Résolution) — {lr_np.shape[1]}×{lr_np.shape[0]}px')
axes[0].axis('off')

axes[1].imshow(hr_np)
axes[1].set_title(f'HR (Haute Résolution) — {hr_np.shape[1]}×{hr_np.shape[0]}px')
axes[1].axis('off')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lr_hr_example.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figure sauvegardée : results/training/lr_hr_example.png")

## 5. Fonctions de perte et métriques

On utilise deux fonctions de perte :
- **L1 Loss** : erreur absolue moyenne — plus robuste que MSE aux outliers, préférable pour la SR
- **PSNR (Peak Signal-to-Noise Ratio)** : métrique standard pour la restauration d'images (en dB)

Un PSNR élevé indique une bonne fidélité de reconstruction.

In [ ]:
def psnr(pred, target, max_val=1.0):
    """Calcule le PSNR entre deux tenseurs."""
    mse = F.mse_loss(pred, target)
    if mse == 0:
        return float('inf')
    return 20 * torch.log10(torch.tensor(max_val)) - 10 * torch.log10(mse)


criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=CONFIG['lr_decay_epochs'], gamma=CONFIG['lr_decay_gamma']
)

print("✅ Critère   : L1 Loss")
print(f"✅ Optimizer : Adam (lr={CONFIG['lr']})")
print(f"✅ Scheduler : MultiStepLR aux époques {CONFIG['lr_decay_epochs']}")

## 6. Boucle d'entraînement

L'entraînement suit la procédure standard :
1. Forward pass : le modèle prédit l'image SR depuis le LR
2. Calcul de la perte L1 entre la prédiction et la vérité terrain (HR)
3. Backpropagation + mise à jour des poids
4. Validation périodique pour monitorer la généralisation
5. Sauvegarde des checkpoints tous les `save_every` époques

In [ ]:
history = {
    'train_loss': [], 'val_loss': [],
    'train_psnr': [], 'val_psnr': [],
    'lr': []
}

best_val_psnr = 0.0
best_epoch    = 0

print(f"🚀 Début de l'entraînement — {CONFIG['num_epochs']} époques")
print("=" * 70)

total_start = time.time()

for epoch in range(1, CONFIG['num_epochs'] + 1):
    epoch_start = time.time()

    # ── ENTRAÎNEMENT ─────────────────────────────────────────────────────────
    model.train()
    train_loss_sum = 0.0
    train_psnr_sum = 0.0
    n_batches_train = 0

    for batch_idx, (lr, hr) in enumerate(train_loader):
        lr = lr.to(DEVICE, non_blocking=True)
        hr = hr.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        sr = model(lr)
        loss = criterion(sr, hr)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        with torch.no_grad():
            p = psnr(sr.clamp(0, 1), hr).item()

        train_loss_sum += loss.item()
        train_psnr_sum += p
        n_batches_train += 1

        if (batch_idx + 1) % CONFIG['log_every'] == 0:
            print(f"   Époque {epoch:>3}/{CONFIG['num_epochs']} "
                  f"| Batch {batch_idx+1:>3}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f} "
                  f"| PSNR: {p:.2f} dB")

    avg_train_loss = train_loss_sum / n_batches_train
    avg_train_psnr = train_psnr_sum / n_batches_train

    # ── VALIDATION ───────────────────────────────────────────────────────────
    model.eval()
    val_loss_sum = 0.0
    val_psnr_sum = 0.0
    n_batches_val = 0

    with torch.no_grad():
        for lr, hr in val_loader:
            lr = lr.to(DEVICE, non_blocking=True)
            hr = hr.to(DEVICE, non_blocking=True)
            sr = model(lr)
            val_loss_sum += criterion(sr, hr).item()
            val_psnr_sum += psnr(sr.clamp(0, 1), hr).item()
            n_batches_val += 1

    avg_val_loss = val_loss_sum / max(n_batches_val, 1)
    avg_val_psnr = val_psnr_sum / max(n_batches_val, 1)

    # ── SCHEDULER ────────────────────────────────────────────────────────────
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # ── HISTORIQUE ───────────────────────────────────────────────────────────
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_psnr'].append(avg_train_psnr)
    history['val_psnr'].append(avg_val_psnr)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    print(f"\n[Époque {epoch:>3}/{CONFIG['num_epochs']}] "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
          f"Train PSNR: {avg_train_psnr:.2f} dB | Val PSNR: {avg_val_psnr:.2f} dB | "
          f"LR: {current_lr:.2e} | {epoch_time:.1f}s")

    # ── SAUVEGARDE DU MEILLEUR MODÈLE ─────────────────────────────────────────
    if avg_val_psnr > best_val_psnr:
        best_val_psnr = avg_val_psnr
        best_epoch    = epoch
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'optim_state' : optimizer.state_dict(),
            'val_psnr'    : avg_val_psnr,
            'config'      : CONFIG,
        }, CKPT_DIR / 'swinir_best.pth')
        print(f"   ⭐ Nouveau meilleur modèle sauvegardé (PSNR val: {avg_val_psnr:.2f} dB)")

    # ── CHECKPOINT PÉRIODIQUE ─────────────────────────────────────────────────
    if epoch % CONFIG['save_every'] == 0:
        torch.save({
            'epoch'       : epoch,
            'model_state' : model.state_dict(),
            'optim_state' : optimizer.state_dict(),
            'val_psnr'    : avg_val_psnr,
            'config'      : CONFIG,
        }, CKPT_DIR / f'swinir_epoch_{epoch:04d}.pth')
        print(f"   💾 Checkpoint sauvegardé : swinir_epoch_{epoch:04d}.pth")

    print("-" * 70)

# ── SAUVEGARDE DU MODÈLE FINAL ────────────────────────────────────────────
torch.save({
    'epoch'       : CONFIG['num_epochs'],
    'model_state' : model.state_dict(),
    'config'      : CONFIG,
}, CKPT_DIR / 'swinir_final.pth')

# Sauvegarde de l'historique
with open(RESULTS_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

total_time = time.time() - total_start
print(f"\n✅ Entraînement terminé en {total_time/60:.1f} minutes.")
print(f"   Meilleur modèle : Époque {best_epoch} — Val PSNR: {best_val_psnr:.2f} dB")
print(f"   Checkpoints dans : {CKPT_DIR}")

## 7. Visualisation des courbes d'entraînement

L'analyse des courbes d'entraînement permet de diagnostiquer :
- La **convergence** du modèle (loss décroissante)
- Le **sur-apprentissage** (overfitting) si la val loss remonte
- L'**effet du scheduler** sur le taux d'apprentissage

Un bon modèle montre une convergence régulière avec un faible écart entre les courbes train et val.

In [ ]:
epochs_range = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Courbes d\'entraînement — SwinIR sur DIV2K', fontsize=14, fontweight='bold')

# Loss d'entraînement et de validation
axes[0, 0].plot(epochs_range, history['train_loss'], label='Train Loss', color='royalblue', linewidth=2)
axes[0, 0].plot(epochs_range, history['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2)
axes[0, 0].set_title('Évolution de la Loss (L1)')
axes[0, 0].set_xlabel('Époque')
axes[0, 0].set_ylabel('L1 Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# PSNR entraînement et validation
axes[0, 1].plot(epochs_range, history['train_psnr'], label='Train PSNR', color='royalblue', linewidth=2)
axes[0, 1].plot(epochs_range, history['val_psnr'],   label='Val PSNR',   color='tomato',    linewidth=2)
axes[0, 1].axhline(best_val_psnr, color='green', linestyle='--', alpha=0.7, label=f'Best: {best_val_psnr:.2f} dB')
axes[0, 1].set_title('Évolution du PSNR (dB)')
axes[0, 1].set_xlabel('Époque')
axes[0, 1].set_ylabel('PSNR (dB)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Taux d'apprentissage
axes[1, 0].plot(epochs_range, history['lr'], color='darkorange', linewidth=2)
axes[1, 0].set_title('Évolution du Learning Rate')
axes[1, 0].set_xlabel('Époque')
axes[1, 0].set_ylabel('LR')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# Écart Train/Val (diagnostic overfitting)
gap_psnr = [t - v for t, v in zip(history['train_psnr'], history['val_psnr'])]
axes[1, 1].fill_between(epochs_range, gap_psnr, alpha=0.4, color='slateblue')
axes[1, 1].plot(epochs_range, gap_psnr, color='slateblue', linewidth=2)
axes[1, 1].axhline(0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].set_title('Écart PSNR Train − Val (diagnostic overfitting)')
axes[1, 1].set_xlabel('Époque')
axes[1, 1].set_ylabel('ΔPSNR (dB)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Figure sauvegardée : results/training/training_curves.png")

In [ ]:
# Résumé final
print("\n" + "═" * 60)
print("           RÉSUMÉ DE L'ENTRAÎNEMENT")
print("═" * 60)
print(f"  Modèle          : SwinIR (embed_dim={CONFIG['embed_dim']})")
print(f"  Dataset         : DIV2K — {len(train_dataset)} patches train")
print(f"  Époques         : {CONFIG['num_epochs']}")
print(f"  Meilleure époque: {best_epoch}")
print(f"  Meilleur PSNR val : {best_val_psnr:.2f} dB")
print(f"  PSNR train final  : {history['train_psnr'][-1]:.2f} dB")
print(f"  Loss train finale : {history['train_loss'][-1]:.4f}")
print(f"  Loss val finale   : {history['val_loss'][-1]:.4f}")
print("═" * 60)
print(f"\n📁 Fichiers générés :")
for f in sorted(CKPT_DIR.iterdir()):
    print(f"   {f}")